In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

Взглянем на макс, мин, средний возраст.

In [2]:
df = pd.read_csv('data/train.csv')

min_age = df['Age'].min()
max_age = df['Age'].max()
mean_age = df['Age'].mean()

print(f"Min age: {min_age}")
print(f"Max age: {max_age}")
print(f"Mean age: {mean_age:.2f}")

Min age: 0.42
Max age: 80.0
Mean age: 29.70


Для заполнения пропусков в Age, одним из подходов будет выявление из Name префиксов и присвоение им среднего возраста из той же категории которой они пренадлежат.

In [3]:
train['Initials'] = train['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
test['Initials'] = test['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

print (train['Initials'].value_counts())

Initials
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Major         2
Mlle          2
Col           2
Don           1
Mme           1
Ms            1
Lady          1
Sir           1
Capt          1
Countess      1
Jonkheer      1
Name: count, dtype: int64


Для каждой новой категории Initials посчитаем медиану по возрасту в группе. 

In [4]:
title_age_median = train.groupby('Initials')['Age'].median()
print(title_age_median)

Initials
Capt        70.0
Col         58.0
Countess    33.0
Don         40.0
Dr          46.5
Jonkheer    38.0
Lady        48.0
Major       48.5
Master       3.5
Miss        21.0
Mlle        24.0
Mme         24.0
Mr          30.0
Mrs         35.0
Ms          28.0
Rev         46.5
Sir         49.0
Name: Age, dtype: float64


Заполняем пропуски

In [5]:
train['Age'] = train['Age'].fillna(train['Initials'].map(title_age_median))
test['Age'] = test['Age'].fillna(train['Initials'].map(title_age_median))

In [6]:
overall_median = train['Age'].median()
# запасной вариант общей медианы для заполнения Age преимущественно для ТЕСТОВОЙ выборки
train['Age'] = train['Age'].fillna(overall_median)
test['Age'] = test['Age'].fillna(overall_median)

Заменяем различные теги на обобщающие.

In [7]:
title_mapping = {
    'Mlle': 'Miss',
    'Ms': 'Miss',
    'MMe': 'Mrs',
    'Dona': 'Mrs',
    'Lady': 'Mrs',
    'Countess': 'Mrs',
    'Capt':'Officer',
    'Col':'Officer',
    'Major':'Officer',
    'Dr':'Officer',
    'Rev':'Officer',
    'Sir': 'Mr',
    'Don': 'Mr',
    'Jonkheer': 'Mr'
}
train['Initials'] = train['Initials'].replace(title_mapping)
test['Initials'] = test['Initials'].replace(title_mapping)

print(train['Initials'].value_counts())

Initials
Mr         520
Miss       185
Mrs        127
Master      40
Officer     18
Mme          1
Name: count, dtype: int64


Просмотр распределний по возрасту - выживанию, исключительно для теста.

In [8]:
age_bins_explore = pd.cut(train['Age'], bins = [0, 5, 12, 18, 25, 35, 50, 65, 100])
survival_by_age = train.groupby(age_bins_explore)['Survived'].mean()
print(train.groupby(age_bins_explore)['Survived'].agg(['mean', 'count']))

               mean  count
Age                       
(0, 5]     0.687500     48
(5, 12]    0.360000     25
(12, 18]   0.428571     70
(18, 25]   0.383838    198
(25, 35]   0.334337    332
(35, 50]   0.396104    154
(50, 65]   0.375000     56
(65, 100]  0.125000      8


Финальное исследование для создания новой фичи.

In [9]:
age_bins_explore = pd.cut(train['Age'], bins = [-0.01, 12, 60, np.inf])
survival_by_age = train.groupby(age_bins_explore)['Survived'].mean()
print(train.groupby(age_bins_explore)['Survived'].agg(['mean', 'count']))

                   mean  count
Age                           
(-0.01, 12.0]  0.575342     73
(12.0, 60.0]   0.370603    796
(60.0, inf]    0.227273     22


In [10]:
bins = [-0.01, 12, 60, np.inf]
labels = [0, 1, 2] # 0 = child, 1 = adult, 2 = senior

train['AgeBin'] = pd.cut(train['Age'], bins = bins, labels = labels).astype(int)
test['AgeBin'] = pd.cut(test['Age'], bins = bins, labels = labels).astype(int)

Survived vs SibSp / Parch (parent + child)

In [11]:
pd.crosstab([train.SibSp],train.Survived).style.background_gradient(cmap='summer_r')


Survived,0,1
SibSp,,
0,398,210
1,97,112
2,15,13
3,12,4
4,15,3
5,5,0
8,7,0


In [12]:
pd.crosstab([train.Parch],train.Survived).style.background_gradient(cmap='summer_r')

Survived,0,1
Parch,,
0,445,233
1,53,65
2,40,40
3,2,3
4,4,0
5,4,1
6,1,0


In [13]:
miss_SibSp = train['SibSp'].isna().sum()
miss_Parch = train['Parch'].isna().sum()
print(miss_SibSp, miss_Parch)

0 0


Объединим Parch + SibSp. '+1' идет для того, чтобы одиночка не получил '0'.

In [14]:
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
test['FamilySize'] = test['SibSp'] + test['Parch'] + 1

Проверяем распределение на новой фиче.

In [15]:
print(train.groupby('FamilySize')['Survived'].agg(['mean', 'count']))

                mean  count
FamilySize                 
1           0.303538    537
2           0.552795    161
3           0.578431    102
4           0.724138     29
5           0.200000     15
6           0.136364     22
7           0.333333     12
8           0.000000      6
11          0.000000      7


Делаем биннинг.

In [16]:
bins = [0, 1, 4, np.inf]
labels = [0, 1, 2] # (0, 1] = (0) Alone, (1, 4] = (1) Small Family, (4, np.inf) = (2) Large Family

train['FamilySizeBin'] = pd.cut(train['FamilySize'], bins=bins, labels=labels).astype(int)
test['FamilySizeBin'] = pd.cut(test['FamilySize'], bins=bins, labels=labels).astype(int)

In [17]:
miss_Fare = train['Fare'].isna().sum()
miss_Fare2 = test['Fare'].isna().sum()
print(miss_Fare, miss_Fare2)

0 1


Обработка Fare. Установление зависимости с PClass, может быть сразу с Embark

In [18]:
fare_bins_explore = pd.qcut(train['Fare'], q=5)
print(train.groupby(fare_bins_explore)['Survived'].agg(['mean', 'count']))

                       mean  count
Fare                              
(-0.001, 7.854]    0.217877    179
(7.854, 10.5]      0.201087    184
(10.5, 21.679]     0.424419    172
(21.679, 39.688]   0.444444    180
(39.688, 512.329]  0.642045    176


In [19]:
train['FareBin'], fare_bin_edges = pd.qcut(train['Fare'], q = 5, retbins=True, labels=[0,1,2,3,4])
fare_bin_edges[0] = -np.inf
fare_bin_edges[-1] = np.inf

fare_median_by_class = train.groupby('Pclass')['Fare'].median()
test['Fare'] = test['Fare'].fillna(test['Pclass'].map(fare_median_by_class))
test['FareBin'] = pd.cut(test['Fare'], bins=fare_bin_edges, labels=[0,1,2,3,4])

In [20]:
pd.crosstab([train.Embarked],train.Pclass).style.background_gradient(cmap='summer_r')

Pclass,1,2,3
Embarked,,,
C,85,17,66
Q,2,3,72
S,127,164,353


In [21]:
pd.crosstab([train.Embarked],train.Survived).style.background_gradient(cmap='summer_r')

Survived,0,1
Embarked,,
C,75,93
Q,47,30
S,427,217


Заполняем пропуски (1) в трейновом датасете модой (классом S - самый частый порт).
Ниже должен быть конструкт для One-hot-encoding для Embarked (беспорядковая фича, три порта отправления),
который будет реализован на этапе построения итогового пайплайна из-за относительной сложности. 

In [22]:
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])

Начинаем рассматривать самую грязную фичу - Cabin.

In [23]:
miss_Cabin = train['Cabin'].isna().sum()
not_miss_Cabin = train['Cabin'].notna().sum()
print(miss_Cabin, not_miss_Cabin)

687 204


In [24]:
train['Deck'] = train['Cabin'].str.extract(r'^([A-Za-z])', expand=False)
print(train['Deck'].value_counts(dropna=False))

Deck
NaN    687
C       59
B       47
D       33
E       32
A       15
F       13
G        4
T        1
Name: count, dtype: int64


In [25]:
pd.crosstab(train['Deck'],train['Pclass']).style.background_gradient(cmap='summer_r')

Pclass,1,2,3
Deck,,,
A,15,0,0
B,47,0,0
C,59,0,0
D,29,4,0
E,25,4,3
F,0,8,5
G,0,0,4
T,1,0,0


In [26]:
pd.crosstab(train['Deck'],train['Survived']).style.background_gradient(cmap='summer_r')

Survived,0,1
Deck,,
A,8,7
B,12,35
C,24,35
D,8,25
E,8,24
F,5,8
G,2,2
T,1,0


Фича Cabin - BAD IDEA! Инициализировать и заполнять пропуски плохая идея, добавим лишь фактор наличия в данных самой каюты.

In [27]:
train['HasCabin'] = train['Cabin'].notna().astype(int)
test['HasCabin'] = test['Cabin'].notna().astype(int)

Небольшая работа по фиче 'Sex'

In [28]:
print(train['Sex'])

0        male
1      female
2      female
3      female
4        male
        ...  
886      male
887    female
888    female
889      male
890      male
Name: Sex, Length: 891, dtype: str


In [29]:
train['Sex'] = train['Sex'].map({'male': 0, 'female': 1})
test['Sex'] = test['Sex'].map({'male': 0, 'female': 1})

Дропаем 'PassengerId', на всякий случай сохраняя его в 'test_ids'.
Дропаем 'Name', 'Ticket', 'Cabin'.

In [30]:
test_ids = test['PassengerId']

train = train.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])
test = test.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])